# CSC 349 HW7: Analysis

These answers use the results from wine_clustering.ipynb with random_state=42. The models use 178 wines and 13 chemical features. The actual cultivar labels are used only in Task 8.

[Open in Colab](https://colab.research.google.com/github/DanielLucero25/CSC349/blob/main/HW7/analysis.ipynb)

## 1. Why feature scaling matters

All three methods use distances between wines. Without scaling, a feature with large numbers can have much more influence than a feature with small numbers. In this dataset proline ranges from 278 to 1680, while hue ranges from 0.48 to 1.71. Magnesium ranges from 70 to 162, while nonflavanoid phenols range from 0.13 to 0.66. Those differences in units do not mean proline and magnesium are automatically more useful.

Euclidean distance squares each feature difference, adds the results and takes the square root. For example, a proline difference of 300 contributes 90,000 to the squared distance. A hue difference of 0.3 contributes only 0.09. Both are reasonable differences within these features, but the proline difference would almost completely control their contribution to the distance. K-Means also directly minimizes squared distances, so this problem strongly affects its centers and assignments.

For DBSCAN and HDBSCAN, distances affect which wines count as close neighbors. Without scaling, wines that are similar in most measurements could look far apart because of proline. Other wines could look close mainly because their proline values match. This can change which groups form and which samples become noise.

StandardScaler subtracts each feature's mean and divides by its standard deviation. After this, one unit means one standard deviation rather than one original measurement unit. The box plots show that the large differences in scale are removed. Scaling gives the features a more balanced role, but it does not guarantee perfect groups. Unusual values and related features can still influence the distances.

## 2. Elbow and silhouette results

Both methods suggested k=3 in my run, which matches the three actual wine cultivars. The elbow estimate uses the largest distance below a line joining the normalized endpoints of the inertia curve. It selected three clusters without using the cultivar labels. Inertia decreased from 2314.00 at k=1 to 1658.76 at k=2 and 1277.93 at k=3. Going from three to four clusters only lowered it to 1175.43. The smaller improvement after three clusters gives the curve its elbow.

The silhouette score was highest at k=3 with about 0.285. The scores at k=2 and k=4 were about 0.259 and 0.260. This suggests that three groups give a better balance between closeness inside groups and separation between groups than the other choices I tested. More clusters lowered inertia, but they did not give a better silhouette score.

There was no disagreement about the best k in this experiment. However, 0.285 is not close to the maximum silhouette of 1, so the groups are not completely separated. Some wines have chemical profiles that overlap. Having three clusters also does not mean every assignment matches the real cultivar. The class count table shows that some class 1 wines were assigned with other cultivars.

The elbow rule is an estimate and can change with the tested range or method used to find the bend. Silhouette measures distance patterns rather than knowledge about wine types. Neither method is guaranteed to recover the real number of classes on every dataset. Here they both support three broad groups in the scaled measurements.

## 3. Comparing the clustering assignments

K-Means created three clusters and assigned all 178 wines. One cluster contained all 59 class 0 wines plus three class 1 wines. Another contained 65 class 1 wines. The last contained all 48 class 2 wines plus three class 1 wines. This was a close match to the cultivars, although cluster numbers themselves do not represent cultivar numbers.

DBSCAN with eps=2.5 and min_samples=10 found two clusters and 32 noise points. I selected this setting using silhouette among tested settings that assigned at least half of the wines and produced at least two clusters. Its larger group contained 56 class 0 wines and 45 class 1 wines. Its other group contained 44 class 2 wines and one class 1 wine. It did not separate the first two cultivars well. The 32 noise points included 25 class 1 wines.

HDBSCAN with min_cluster_size=5 and min_samples=5 also found two clusters, but it marked 58 wines as noise. Its larger group combined 53 class 0 wines and 33 class 1 wines. Its smaller group contained 33 class 2 wines and one class 1 wine. It kept fewer samples than DBSCAN under these settings. This shows that using a hierarchy does not guarantee finding all three cultivars.

For example, rows 13 and 25 were noise in both density methods. For DBSCAN, a noise point has too few nearby samples to be a core point and is not within eps of a core point. Such a wine cannot join a dense group under the chosen settings. For HDBSCAN, a sample can be left out if it does not belong to one of the stable groups that survives the hierarchy selection. Noise does not prove that the wine is incorrect or chemically invalid. It means the algorithm did not place it in a supported group under these settings.

K-Means has no noise option. Even an isolated or unusual wine must go to the nearest center. This gives full coverage but can hide unusual samples. Density methods can show those samples separately, although their choices depend on the settings. The PCA plots only keep about 55.41 percent of the variance, so a noise point can look close to a group in the plot while being farther away across all 13 features.

## 4. How DBSCAN and HDBSCAN work

DBSCAN uses one eps radius for the entire dataset. A core point has at least min_samples points inside that radius, counting itself. Connected core points form groups. Border points can join a group by being close to a core point even if they do not have enough neighbors themselves. Points that cannot join are noise.

One radius can be a problem when groups have different densities. A small radius may work for a tight group but break apart a more spread out group. A large radius may keep the spread out group but also connect groups that should stay separate. My parameter table shows this sensitivity. With eps=1.5 and min_samples=5, all 178 wines were noise. With eps=3.0 and min_samples=5, the model found just one cluster and 11 noise points.

HDBSCAN builds a hierarchy across density levels instead of choosing one global eps. It first uses a local core distance based on nearby neighbors. The mutual reachability distance between two points is the largest of their ordinary distance and their two core distances. This makes connections through sparse areas more expensive. The method builds a hierarchy, removes branches that are too small and selects groups that stay stable across density levels.

This approach can keep groups with different local densities without forcing the same radius on all of them. However, it still depends on settings such as min_cluster_size and min_samples. It can still merge cultivars or mark many samples as noise, as it did here.

The Task 7 dendrogram helps show how small groups merge into larger ones as the allowed distance increases. It uses ordinary Euclidean single linkage, so it is not the exact HDBSCAN tree. HDBSCAN uses mutual reachability and stability selection instead of simply cutting that ordinary dendrogram at one height.

## 5. Using clusters to predict the wine cultivar

I trained a separate logistic regression model for each algorithm using only its cluster assignments as an input. I used one-hot encoding because the cluster numbers are categories, not measurements with a meaningful order. The training labels let the classifier learn the connection between those categories and the actual cultivars. All three models used the same stratified split with 142 training wines and 36 test wines.

| Clustering method | Correct test predictions | Test accuracy | Silhouette excluding noise | Wines assigned |
| --- | --- | --- | --- | --- |
| K-Means | 35 of 36 | 97.22 percent | 0.285 | 178 of 178 |
| DBSCAN | 24 of 36 | 66.67 percent | 0.323 | 146 of 178 |
| HDBSCAN | 27 of 36 | 75.00 percent | 0.352 | 120 of 178 |

K-Means gave the most useful representation for predicting cultivars in this run. It made only one test error, on a class 1 wine. Its groups closely matched the real classes, so the classifier could learn a useful mapping. DBSCAN made 12 errors, including 11 class 1 wines predicted as class 0. HDBSCAN made nine errors. Both density methods merged many class 0 and class 1 wines, which removed information needed to tell them apart. The majority-class baseline was only 38.89 percent, so all three representations still carried useful information.

A classifier receiving only a cluster category cannot tell apart two wines with the same category. It has no access to their original chemical differences. Noise makes this harder because all samples labeled -1 share one category even though they may belong to several cultivars and may not resemble each other. I kept noise in the classification experiment so the reported accuracy includes these difficult cases.

The density methods had higher silhouette scores, but this did not lead to higher cultivar accuracy. Their silhouette scores exclude noise and measure only the wines they kept. HDBSCAN kept about 67.42 percent of the wines, compared with 82.02 percent for DBSCAN and 100 percent for K-Means. Removing difficult samples can improve silhouette while losing useful class information. Silhouette also measures geometric separation rather than whether clusters match cultivars.

This experiment reused the full-data clustering assignments from the earlier tasks. The test cultivar labels were not used to fit the classifier, but the test feature rows had already helped fit scaling and clustering. This is a transductive comparison, so the accuracy is not a clean estimate for completely unseen wines. A future-data test should split before fitting the scaler and cluster models and use a defined method for assigning new wines. The test set is also small, so one correct or incorrect wine changes accuracy by about 2.78 percentage points. These results support K-Means for this comparison, but they do not prove that it will always perform best.

## References

[Scikit-Learn Wine dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)

[Scikit-Learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html)

[Scikit-Learn HDBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)